# Benchmark

Saved runs held against a model trained on the subjects' own answers, subject by subject, one block per benchmark. The first cell is the only setup: name the topic and the runs (each by any text only its stem contains), the name the written files carry, whether anything is written, and whether figures use the topic's `plot.json`. `PREDICTORS` names the columns the trained model is given (a list); `None` takes the dataset's grouping columns. `python benchmark.py <topic> <run> ...` runs the same blocks in one go; `RUN_NAMES` may also be a mapping, label -> text, and the runs are then called by those labels; `load_comparisons(paths.CONFIG_DIR / "comparisons.json")["haiku"].runs` is such a mapping, a named set. `python analyze.py <topic>` lists a topic's runs.

Only subjects whose own answer is known and that every named run answered are scored. The trained model is a ridge logit on the predictors, cross-validated, so nobody's probabilities come from a fit that saw their answer.

In [ ]:
%matplotlib inline
from panel import paths
from panel.analysis import Output
from panel.analysis.benchmark import agreement, calibration, describe, knowledge, load_ground, scores
from panel.analysis.comparison import load_runs
from panel.run import ResultStore

TOPIC = "election_2021"
RUN_NAMES = ["sod_default_cs_default_AnthropicLegacy_claude-haiku-4-5", "sod_default_cs_default_Anthropic_claude-haiku-5-5"]
NAME = TOPIC
SAVE = False
PREDICTORS = None
STYLED = True

STORE = ResultStore(paths.RESULTS_DIR, TOPIC)
GROUND = load_ground(load_runs(STORE, RUN_NAMES, styled=STYLED), PREDICTORS)
OUT = Output(STORE, NAME, save=SAVE)
print(describe(GROUND))

## Scores on the subjects' own answers

Log-loss and Brier score, lower is better. `claimed shares` gives everybody the same distribution, the shares of the answers; `trained on claims` is the trained model. Per run: its answers as given, its mean answer given to everybody, and its answers after recalibration (a cross-validated logit on their logarithms).

In [ ]:
table, figures = scores(GROUND, OUT)
display(table.round(3), *figures)

## What a run knows of a subject

The conditioning gain is how much the subject's description improved on the run's mean answer. The knowledge ratio is how far the recalibrated answers get from the claimed shares towards the trained model: 1 is as much as that model. It is empty when the trained model itself does not beat the claimed shares, as on a small sample.

In [ ]:
table, figures = knowledge(GROUND, OUT)
display(table.round(3), *figures)

## Agreement with the trained model, outcome by outcome

Over the subjects: the correlation of a run's probability with the trained model's, and the ratio of their standard deviations.

In [ ]:
table, figures = agreement(GROUND, OUT)
display(table.round(3), *figures)

## Calibration

Probabilities pooled over the outcomes and binned by size; a bin's mean probability against how often its outcome was the answer.

In [ ]:
table, figures = calibration(GROUND, OUT)
display(table.round(3), *figures)